# Step 1: Integration and Main Cell Type Labeling

This step processes and integrates multiple Xenium samples for downstream analysis.

**Reference:** Based on methodology from Yu et al. (2025) https://doi.org/10.1038/s41588-025-02158-6

## Setup and imports

In [ ]:
# Imports
import scanpy as sc
import squidpy as sq
from spatialdata_io import xenium
import spatialdata as sd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
import anndata as ad
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import glob
import os
import shutil

In [ ]:
# Reproducibility settings
sc.settings.verbosity = 3
np.random.seed(26)

In [ ]:
# Path config
indir = '/Users/yyj/Doc/1_dod_dec25/raw_data/' #indir = '/path/to/xenium/raw_data'
outdir = '/Users/yyj/Doc/1_dod_dec25/processed_data/' #outdir = '/path/to/integrated/processed_data'
os.makedirs(outdir, exist_ok=True)
os.makedirs('figures/integrated/', exist_ok=True)

## Helper functions

In [ ]:
## Xenium Morphology Focus Naming Fix (if needed)

def fix_morphology_focus_naming(xenium_path):
    """
    Fix morphology focus naming by temporarily hiding original files.
    Required for spatialdata_io.xenium() compatibility.
    """
    morphology_dir = os.path.join(xenium_path, "morphology_focus")
    
    if not os.path.exists(morphology_dir):
        print("Morphology focus directory not found - skipping fix")
        return False
    
    # Create a backup directory
    backup_dir = os.path.join(morphology_dir, "backup_original")
    os.makedirs(backup_dir, exist_ok=True)
    
    # Move original files to backup
    original_files = [
        'ch0000_dapi.ome.tif',
        'ch0001_atp1a1_cd45_e-cadherin.ome.tif', 
        'ch0002_18s.ome.tif',
        'ch0003_alphasma_vimentin.ome.tif'
    ]
    
    for orig_file in original_files:
        orig_path = os.path.join(morphology_dir, orig_file)
        backup_path = os.path.join(backup_dir, orig_file)
        
        if os.path.exists(orig_path):
            shutil.move(orig_path, backup_path)
            print(f"Moved to backup: {orig_file}")
    
    # Create symbolic links
    links = {
        'morphology_focus_0000.ome.tif': 'backup_original/ch0000_dapi.ome.tif',
        'morphology_focus_0001.ome.tif': 'backup_original/ch0001_atp1a1_cd45_e-cadherin.ome.tif',
        'morphology_focus_0002.ome.tif': 'backup_original/ch0002_18s.ome.tif',
        'morphology_focus_0003.ome.tif': 'backup_original/ch0003_alphasma_vimentin.ome.tif'
    }
    
    for link_name, target_path in links.items():
        link_path = os.path.join(morphology_dir, link_name)
        full_target_path = os.path.join(morphology_dir, target_path)
        
        if os.path.exists(full_target_path) and not os.path.exists(link_path):
            os.symlink(target_path, link_path)
            print(f"Created link: {link_name} → {target_path}")
    
    return True

def restore_morphology_focus_naming(xenium_path):
    """
    Restore original morphology focus files (optional cleanup).
    """
    morphology_dir = os.path.join(xenium_path, "morphology_focus")
    backup_dir = os.path.join(morphology_dir, "backup_original")
    
    if os.path.exists(backup_dir):
        # Remove symbolic links
        for i in range(4):
            link_path = os.path.join(morphology_dir, f"morphology_focus_000{i}.ome.tif")
            if os.path.islink(link_path):
                os.unlink(link_path)
                print(f"Removed link: morphology_focus_000{i}.ome.tif")
        
        # Move files back
        for orig_file in os.listdir(backup_dir):
            orig_path = os.path.join(backup_dir, orig_file)
            restore_path = os.path.join(morphology_dir, orig_file)
            shutil.move(orig_path, restore_path)
            print(f"Restored: {orig_file}")
        
        # Remove backup directory
        shutil.rmtree(backup_dir)
        print("Restored original morphology focus files")

## Load individual samples
Saving individual samples as .h5ad

In [ ]:
# Load and save individual samples (loop over all samples found in indir)

sample_dirs = sorted([p for p in glob.glob(os.path.join(indir, "*")) if os.path.isdir(p)])

if len(sample_dirs) == 0:
    raise ValueError(f"No sample directories found in {indir}")

for xenium_path in sample_dirs:
    sampleName = os.path.basename(xenium_path.rstrip("/"))
    print(f"Processing {sampleName}...")

    # apply naming fix when needed
    fix_morphology_focus_naming(xenium_path) # if needed

    sdata = xenium(xenium_path)
    adata = sdata.tables["table"]

    out_file = os.path.join(outdir, f"anndata_{sampleName}.h5ad")
    adata.write_h5ad(out_file, compression="gzip")
    print(f"Saved: {out_file}")

## Combine samples


In [ ]:
# Load all preprocessed samples
annFileArray = sorted(glob.glob(outdir + "anndata_*.h5ad"))
ann_list = []

for filename in annFileArray:
    print(f"Loading: {filename}")
    ann = sc.read_h5ad(filename)
    sampleName = Path(filename).stem.replace('anndata_', '')
    ann.obs['sampleName'] = sampleName
    ann_list.append(ann)


In [ ]:
# Concatenate all samples
sample_keys = [Path(f).stem.replace('anndata_', '') for f in annFileArray]
ann_comb = ad.concat(
    ann_list, 
    join='outer', 
    axis=0,
    label="sample", 
    keys=sample_keys,
    merge='same'
)

print(f"Combined dataset: {ann_comb.n_obs} cells × {ann_comb.n_vars} features")

## Feature selection


In [ ]:
# Filter out mutation calls and wild-type probes
features = ann_comb.var.index
sele_features = features[~features.str.contains(r'>|_WT')].tolist()
ann_comb.var['is_gene'] = ~ann_comb.var_names.str.contains(r'>|_WT|clone')

print(f"Selected features: {len(sele_features)} / {len(features)}")


## Quality control


In [ ]:
# Calculate QC metrics
sc.pp.calculate_qc_metrics(
    ann_comb, 
    percent_top=(10, 20, 50, 150),
    inplace=True
)


In [ ]:
# Calculate QC metrics using only gene probes
sc.pp.calculate_qc_metrics(
    ann_comb, 
    percent_top=(10, 20, 50, 150), 
    inplace=True,
    qc_vars=['is_gene']
)

In [ ]:
# Report negative control percentages
cprobes = (ann_comb.obs["control_probe_counts"].sum() / ann_comb.obs["total_counts"].sum() * 100)
cwords = (ann_comb.obs["control_codeword_counts"].sum() / ann_comb.obs["total_counts"].sum() * 100)
print(f"Negative DNA probe count %: {cprobes:.2f}")
print(f"Negative decoding count %: {cwords:.2f}")

In [ ]:
# Visualize QC metrics
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(ann_comb.obs["total_counts"], kde=False, ax=axs[0])
axs[0].set_xlim(0, 200)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(ann_comb.obs["n_genes_by_counts"], kde=False, ax=axs[1])

axs[2].set_title("Area of segmented cells")
sns.histplot(ann_comb.obs["cell_area"], kde=False, ax=axs[2])

axs[3].set_title("Nucleus ratio")
sns.histplot(ann_comb.obs["nucleus_area"] / ann_comb.obs["cell_area"], kde=False, ax=axs[3])

plt.tight_layout()
plt.show()

## Filtering and normalization


In [ ]:
# Store raw counts
ann_comb.layers["raw_counts"] = ann_comb.X.copy()

In [ ]:
# Filter cells and genes
# Thresholds based on QC distributions (min_counts=100, min_cells=30)
print(f"Before filtering: {ann_comb.n_obs} cells, {ann_comb.n_vars} genes")
sc.pp.filter_cells(ann_comb, min_counts=20)
sc.pp.filter_genes(ann_comb, min_cells=30)
print(f"After filtering: {ann_comb.n_obs} cells, {ann_comb.n_vars} genes")

In [ ]:
# Normalize and log-transform using Scanpy's standard method
sc.pp.normalize_total(ann_comb, target_sum=1e4, inplace=True)
sc.pp.log1p(ann_comb)

In [ ]:
# Set highly variable genes based on selected features
ann_comb.var["highly_variable"] = ann_comb.var.index.isin(sele_features)
print(f"Highly variable genes: {ann_comb.var['highly_variable'].sum()}")

## Spatial QC and coordinate verification


In [ ]:
# Verify spatial coordinates exist and are valid
assert 'spatial' in ann_comb.obsm, "Missing 'spatial' coordinates in obsm"
coords = ann_comb.obsm['spatial']

print("=== Spatial Coordinate Verification ===")
print(f"Total cells: {ann_comb.n_obs}")
print(f"Coordinate shape: {coords.shape}")
print(f"Coordinate range X: [{coords[:, 0].min():.1f}, {coords[:, 0].max():.1f}]")
print(f"Coordinate range Y: [{coords[:, 1].min():.1f}, {coords[:, 1].max():.1f}]")

In [ ]:
# Check for duplicate/invalid coordinates
duplicates = pd.DataFrame(coords).duplicated().sum()
nan_coords = np.isnan(coords).any(axis=1).sum()
inf_coords = np.isinf(coords).any(axis=1).sum()
print(f"Duplicate coordinates: {duplicates} ({duplicates/ann_comb.n_obs*100:.2f}%)")
print(f"NaN coordinates: {nan_coords}")
print(f"Inf coordinates: {inf_coords}")

In [ ]:
# Verify coordinates per sample
print("\n=== Per-Sample Coordinate Verification ===")
for sample in sorted(ann_comb.obs['sample'].unique()):
    sample_mask = ann_comb.obs['sample'] == sample
    sample_coords = coords[sample_mask]
    area = (sample_coords[:, 0].max() - sample_coords[:, 0].min()) * \
           (sample_coords[:, 1].max() - sample_coords[:, 1].min())
    density = sample_mask.sum() / area if area > 0 else 0
    print(f"{sample}: {sample_mask.sum()} cells, density: {density:.2f} cells/unit²")

In [ ]:
# Visualize spatial distribution per sample
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, sample in enumerate(sorted(ann_comb.obs['sample'].unique())):
    sample_mask = ann_comb.obs['sample'] == sample
    sample_coords = coords[sample_mask]
    axes[i].scatter(sample_coords[:, 0], sample_coords[:, 1], s=0.5, alpha=0.3)
    axes[i].set_title(f'{sample}\n(n={sample_mask.sum()})')
    axes[i].set_aspect('equal')
    axes[i].axis('off')

plt.suptitle('Spatial Distribution per Sample', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Check spatial cell density distribution
nbrs = NearestNeighbors(n_neighbors=2).fit(coords)
distances, _ = nbrs.kneighbors(coords)
nn_distances = distances[:, 1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(nn_distances, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(np.median(nn_distances), color='r', linestyle='--', 
           label=f'Median: {np.median(nn_distances):.2f}')
ax.set_xlabel('Distance to Nearest Neighbor')
ax.set_ylabel('Frequency')
ax.set_title('Spatial Cell Density Distribution')
ax.legend()
plt.show()

print(f"Mean nearest neighbor distance: {nn_distances.mean():.2f}")
print(f"Median nearest neighbor distance: {np.median(nn_distances):.2f}")

## Integration and batch correction


In [ ]:
# PCA and initial UMAP (before batch correction)
sc.pp.pca(ann_comb, n_comps=N_PCS, random_state=RANDOM_STATE)
sc.pp.neighbors(ann_comb, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS, random_state=RANDOM_STATE)
sc.tl.umap(ann_comb, min_dist=UMAP_MIN_DIST, random_state=RANDOM_STATE)

In [ ]:
# Harmony integration for batch correction
sc.external.pp.harmony_integrate(ann_comb, 'sample')

In [ ]:
# Recompute neighbors and UMAP using Harmony-corrected PCA
sc.pp.neighbors(ann_comb, use_rep="X_pca_harmony_use", n_neighbors=N_NEIGHBORS, random_state=RANDOM_STATE)
sc.tl.umap(ann_comb, min_dist=UMAP_MIN_DIST, random_state=RANDOM_STATE)

In [ ]:
# Clustering
sc.tl.leiden(ann_comb, resolution=0.8, key_added='leiden_0.8')

In [ ]:
# Visualize integration results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(ann_comb, color='sample', ax=axes[0], show=False, title='After Harmony Integration')
sc.pl.umap(ann_comb, color='leiden_0.8', ax=axes[1], show=False, title='Leiden Clusters (res=0.8)')
plt.tight_layout()
plt.show()

## Batch correction validation


In [ ]:
# Sample size parameter
sample_size = 50000

In [ ]:
sil_before = silhouette_score(
    ann_comb.obsm['X_pca'][:, :10], 
    ann_comb.obs['sample'].astype('category').cat.codes,
    sample_size=sample_size  # sklearn will sample internally
)

In [ ]:
sil_after = silhouette_score(
    ann_comb.obsm['X_pca_harmony'][:, :10], 
    ann_comb.obs['sample'].astype('category').cat.codes,
    sample_size=sample_size
)

In [ ]:
print(f"Silhouette Score (by batch):")
print(f"  Before Harmony: {sil_before:.3f}")
print(f"  After Harmony: {sil_after:.3f}")
print(f"  Improvement: {sil_before - sil_after:.3f}")

## Cell type annotation


In [ ]:
# Differential expression analysis for cluster annotation
sc.pl.violin(
    ann_comb, 
    ['total_counts', 'total_counts_is_gene'], 
    groupby="leiden_0.8", 
    size=0
)


In [ ]:
# Get top marker genes
sc.tl.rank_genes_groups(
    ann_comb, 
    'leiden_0.8', 
    method='wilcoxon', 
    key_added="wilcoxon"
)

# Visualize
sc.pl.rank_genes_groups(
    ann_comb, 
    n_genes=10, 
    sharey=False, 
    key="wilcoxon"
)

In [ ]:
# Extract and save marker genes
df_genes = pd.DataFrame(ann_comb.uns['wilcoxon']['names'])
df_genes.to_csv(outdir + '/main_celltypemarkers_wilcoxon_08.csv', index=False)

In [ ]:
# Check current number of cells
print(f"Before filtering: {ann_comb.n_obs} cells")
print(f"Cells in cluster 4: {(ann_comb.obs['leiden_0.8'] == '4').sum()}")

# Remove cluster 4
ann_comb = ann_comb[ann_comb.obs['leiden_0.8'] != '4'].copy()

print(f"After filtering: {ann_comb.n_obs} cells")

In [ ]:
# Visualize integration results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(ann_comb, color='sample', ax=axes[0], show=False, title='After Harmony Integration')
sc.pl.umap(ann_comb, color='leiden_0.8', ax=axes[1], show=False, title='Leiden Clusters (res=0.8)')
plt.tight_layout()
plt.show()

In [ ]:
# Annotate cell types based on marker gene expression
# Annotation mapping based on marker gene analysis
ann_comb.obs["celltype"] = ann_comb.obs["leiden_0.8"].map(
    {
        "0": "Neuroblast",
        "1": "Neuroblast", 
        "2": "Neuroblast",
        "3": "Neuroblast",
        "5": "T",
        "6": "Fibroblast",
        "7": "Neuroblast",
        "8": "Macrophage",
        "9": "Endothelial",
        "10": "Macrophage",
        "11": "Schwann",
        "12": "B",
        "13": "Neuroblast",
        "14": "Neuroblast"
    }
)


In [ ]:
# Visualize cell type annotation
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
sc.pl.umap(ann_comb, color='celltype', ax=ax, show=False, title='Cell Type Annotation', legend_loc='none', size=5)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize cell type density
sc.tl.embedding_density(ann_comb, groupby="celltype")
sc.pl.embedding_density(ann_comb, groupby="celltype") 

## Subclustering of major cell types


Subclustering resolutions were chosen after testing multiple resolutions (0.1-0.7) as it provided optimal balance between:
- Sufficient granularity to distinguish subtypes
- Biological interpretability (clear separation and markers)
- Avoidance of over-clustering (too many small clusters) or under-clustering (merged subtypes)


In [ ]:
# Subcluster major cell types at optimized resolutions
# Resolutions chosen based on biological relevance and cluster stability
celltypes = ["Neuroblast", "Macrophage", "B", "T"]
resolutions = {"Neuroblast": 0.7, "Macrophage": 0.3, "B": 0.3, "T": 0.5}

RANDOM_STATE = 26
N_PCS = 20
N_PCS_SUBTYPE = 15
N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.2

In [ ]:
for ct in celltypes:
    print(f"\nSubclustering {ct}...")
    
    # Subset to cell type
    ad_sub = ann_comb[ann_comb.obs["cell_type"] == ct].copy()
    
    if ad_sub.n_obs < 100:
        print(f"  Skipping {ct}: insufficient cells ({ad_sub.n_obs})")
        continue
    
    # Recompute HVG, PCA, neighbors for subclustering
    sc.pp.pca(ad_sub, n_comps=N_PCS_SUBTYPE, random_state=RANDOM_STATE)
    sc.pp.neighbors(ad_sub, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS_SUBTYPE, random_state=RANDOM_STATE)
    sc.tl.umap(ad_sub, min_dist=UMAP_MIN_DIST, random_state=RANDOM_STATE)

    
    # Cluster at specified resolution
    res = resolutions[ct]
    key = f"leiden_{ct}_{res}"
    sc.tl.leiden(ad_sub, resolution=res, key_added=key)
    
    print(f"  Found {ad_sub.obs[key].nunique()} subclusters at resolution {res}")
    
    # Save subclustered data
    ad_sub.write_h5ad(
        outdir + f"xenium_{ct}_sub.h5ad",
        compression="gzip"
    )
    print(f"  Saved: {ct}_subclustered.h5ad")

print("\nSubclustering complete!")

In [ ]:
# Save integrated dataset
ann_comb.write_h5ad(outdir + 'xenium_integrated.h5ad', compression='gzip')
print(f"Saved integrated dataset: {ann_comb.n_obs} cells × {ann_comb.n_vars} features")